# Statistical Comparison Plan — Paired Significance Tests

In [ ]:
import numpy as np
from scipy import stats

per_seed = {
    "text": {
        "vanilla": [0.560, 0.670, 0.687, 0.560, 0.643],
        "fedtgp":  [0.627, 0.627, 0.687, 0.697, 0.600],
    },
    "audio": {
        "vanilla": [0.652, 0.552, 0.526, 0.574, 0.574],
        "fedtgp":  [0.500, 0.472, 0.574, 0.578, 0.574],
    },
}

# sanity check: means/SDs should reproduce the thesis tables
for silo, d in per_seed.items():
    for cfg, vals in d.items():
        print(f"{silo:5s} {cfg:8s}  mean={np.mean(vals):.3f}  sd={np.std(vals):.3f}")

## Paired tests: FedTGP-style vs. Vanilla FedProto

In [ ]:
def paired_report(a, b, name_a, name_b, alpha=0.05):
    a, b = np.asarray(a, float), np.asarray(b, float)
    diff = a - b
    mean_diff = diff.mean()

    # normality of the differences (Shapiro); needs variation to be defined
    if len(np.unique(diff)) > 1:
        shapiro_p = stats.shapiro(diff).pvalue
    else:
        shapiro_p = np.nan

    # paired t-test
    t_stat, t_p = stats.ttest_rel(a, b)

    # Wilcoxon signed-rank (guard: undefined if all diffs are zero)
    if np.count_nonzero(diff) == 0:
        w_stat, w_p = np.nan, np.nan
    else:
        w_stat, w_p = stats.wilcoxon(a, b)

    use_t = (not np.isnan(shapiro_p)) and (shapiro_p > 0.05)
    primary = ("paired t-test", t_p) if use_t else ("Wilcoxon", w_p)

    print(f"  {name_a} vs {name_b}")
    print(f"    mean paired diff ({name_a} - {name_b}) = {mean_diff:+.4f}")
    print(f"    Shapiro-Wilk p (normality of diffs)    = {shapiro_p:.4f}"
          + ("  -> differences look normal" if use_t else "  -> use Wilcoxon"))
    print(f"    paired t-test:            t = {t_stat:+.3f},  p = {t_p:.4f}")
    print(f"    Wilcoxon signed-rank:     W = {w_stat},  p = {w_p:.4f}")
    print(f"    PRIMARY ({primary[0]}): p = {primary[1]:.4f}  -> "
          + ("SIGNIFICANT" if primary[1] < alpha else f"not significant at alpha={alpha}"))
    print()
    return {"silo_pair": f"{name_a}_vs_{name_b}", "mean_diff": mean_diff,
            "shapiro_p": shapiro_p, "t_p": t_p, "wilcoxon_p": w_p,
            "primary_test": primary[0], "primary_p": primary[1]}

results = []
for silo, d in per_seed.items():
    print(f"===== {silo.upper()} =====")
    results.append({"silo": silo,
                    **paired_report(d["fedtgp"], d["vanilla"], "FedTGP", "Vanilla")})

## Summary table

In [ ]:
import pandas as pd
summary = pd.DataFrame(results)[
    ["silo", "mean_diff", "shapiro_p", "t_p", "wilcoxon_p", "primary_test", "primary_p"]
]
summary = summary.round(4)
print(summary.to_string(index=False))

print()
print("Note (n=5): the two-sided Wilcoxon signed-rank test has a minimum attainable")
print("p-value of 0.0625 at 5 pairs, so it cannot reach p<0.05 regardless of effect size.")
print("Differences are reported as descriptive, per Section 2.10.7.")

# optional: save
summary.to_csv("statistical_tests_summary.csv", index=False)
print("\nsaved: statistical_tests_summary.csv")

## Optional: federated vs. standalone

In [ ]:
standalone = {"text": 0.670, "audio": 0.710}   # Table 3.1 point estimates

for silo in ["text", "audio"]:
    for cfg in ["vanilla", "fedtgp"]:
        vals = np.array(per_seed[silo][cfg], float)
        t_stat, t_p = stats.ttest_1samp(vals, standalone[silo])
        print(f"{silo:5s} {cfg:8s} vs standalone {standalone[silo]:.3f}: "
              f"mean={vals.mean():.3f}, t={t_stat:+.3f}, p={t_p:.4f} "
              + ("(SIGNIFICANT)" if t_p < 0.05 else "(n.s.)"))

## Figure: per-seed paired comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

silos = list(per_seed.keys())          # ["text", "audio"]
n_seeds = len(next(iter(per_seed.values()))["vanilla"])

fig, axes = plt.subplots(1, len(silos), figsize=(4.2 * len(silos), 4.2), sharey=True)
if len(silos) == 1:
    axes = [axes]

x = [0, 1]
for ax, silo in zip(axes, silos):
    v = np.array(per_seed[silo]["vanilla"], float)
    t = np.array(per_seed[silo]["fedtgp"], float)

    # one connecting line per seed
    for s in range(n_seeds):
        ax.plot(x, [v[s], t[s]], marker="o", color="gray", alpha=0.7,
                linewidth=1.2, markersize=5,
                label=f"seed {s}" if silo == silos[0] else None)

    # mean markers
    ax.plot(x, [v.mean(), t.mean()], marker="D", color="crimson",
            linewidth=2.5, markersize=9, zorder=5,
            label="mean" if silo == silos[0] else None)

    # standalone floor if available
    floor = {"text": 0.670, "audio": 0.710}.get(silo)
    if floor is not None:
        ax.axhline(floor, ls="--", lw=1.0, color="steelblue", alpha=0.8)
        ax.text(1.02, floor, "standalone", va="center", ha="left",
                fontsize=8, color="steelblue")

    ax.set_xticks(x)
    ax.set_xticklabels(["Vanilla\nFedProto", "FedTGP\nstyle"])
    ax.set_xlim(-0.35, 1.55)
    ax.set_title(f"{silo.upper()}  (n.s., 5 seeds)")
    ax.grid(axis="y", alpha=0.3)

axes[0].set_ylabel("Test macro-F1")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=n_seeds + 1,
           bbox_to_anchor=(0.5, -0.08), fontsize=8, frameon=False)
fig.suptitle("Per-seed paired comparison: Vanilla FedProto vs FedTGP-style", y=1.02)
fig.tight_layout()
fig.savefig("paired_seed_comparison.png", dpi=200, bbox_inches="tight")
plt.show()
print("saved: paired_seed_comparison.png")